# 05 Figures: metrics and capture curve

Drafted with an AI assistant, see docs/ai_log.md.

Inputs: `data/processed/model_table.parquet`, `data/processed/predictions.parquet`.
Outputs: `figures/metrics.csv`, `figures/capture_curve.pdf`.

Metrics per model, split and label: capture at 5% and 10%, ROC AUC, average precision. A `random` row gives the reference values.

In [ ]:
import helpers
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.metrics import average_precision_score, roc_auc_score

In [ ]:
model_table = pd.read_parquet(helpers.PROCESSED / "model_table.parquet")
predictions = pd.read_parquet(helpers.PROCESSED / "predictions.parquet")
data = model_table.merge(predictions, on=["egid", "cutoff", "split"], how="inner")
assert len(data) == len(predictions)
helpers.FIGURES.mkdir(exist_ok=True)

One row per (model, split, label). Labels: `y` for both splits, `y_2024` for the test split.

In [ ]:
rows = []
for split in ["train", "test"]:
    labels = ["y"] if split == "train" else ["y", "y_2024"]
    for label in labels:
        for model, part in data[data["split"] == split].groupby("model"):
            y = part[label].astype(int).to_numpy()
            score = part["score"].to_numpy()
            rows.append(
                {
                    "model": model,
                    "split": split,
                    "label": label,
                    "n": len(y),
                    "positives": int(y.sum()),
                    "base_rate": y.mean(),
                    "capture_5": helpers.capture_at_k(y, score, 0.05),
                    "capture_10": helpers.capture_at_k(y, score, 0.10),
                    "roc_auc": roc_auc_score(y, score),
                    "avg_precision": average_precision_score(y, score),
                }
            )
        # Random reference: same pool, expected values
        rows.append(
            {
                "model": "random",
                "split": split,
                "label": label,
                "n": rows[-1]["n"],
                "positives": rows[-1]["positives"],
                "base_rate": rows[-1]["base_rate"],
                "capture_5": 0.05,
                "capture_10": 0.10,
                "roc_auc": 0.5,
                "avg_precision": rows[-1]["base_rate"],
            }
        )
metrics = pd.DataFrame(rows).sort_values(["split", "label", "model"]).reset_index(drop=True)
metrics = metrics.round(6)
metrics.to_csv(helpers.FIGURES / "metrics.csv", index=False, lineterminator="\n")
print(metrics.to_string())

Capture curve on the test split, label `y`: share of adopters found as we go down the ranking.

In [ ]:
test = data[data["split"] == "test"]
fig, ax = plt.subplots(figsize=(5, 4))
for model, part in test.groupby("model"):
    y = part["y"].astype(int).to_numpy()
    order = np.argsort(-part["score"].to_numpy(), kind="stable")
    share_ranked = np.arange(1, len(y) + 1) / len(y)
    share_captured = np.cumsum(y[order]) / y.sum()
    ax.plot(share_ranked, share_captured, label=model)
ax.plot([0, 1], [0, 1], linestyle="--", color="grey", label="random")
ax.set_xlabel("Share of buildings ranked (highest score first)")
ax.set_ylabel("Share of adopters captured")
ax.set_title("Capture curve, test split (first PV 2024-2025)")
ax.legend(loc="lower right")
fig.tight_layout()
fig.savefig(helpers.FIGURES / "capture_curve.pdf", metadata={"CreationDate": None})
plt.close(fig)